In [1]:
import pandas as pd
import numpy as np

In [2]:
df= pd.read_csv('../03.results/outputs/ATOM_USD_ML_features_preprocessed.csv')
df.head()

,last_price,open,high,low,volume,last_return,return_3d,returnvola_3d,volume_3d,volumevola_3d,...,bb_upper_20,bb_lower_20,bb_width_20,high_low_range_pct,day_of_week,month,Surge,target_next_close,target_next_return,target_direction
0,5.915,4.913,5.985,4.913,6110000.0,0.203704,0.121114,0.152823,17400000.0,1.450069e+06,...,5.528342,2.892258,0.626104,0.218197,2,8,1,6.076,0.027219,1.0
1,6.076,5.907,6.304,5.495,7730000.0,0.027219,0.110989,0.153215,18060000.0,1.756730e+06,...,5.867212,2.779188,0.714291,0.136956,3,8,1,6.096,0.003292,1.0
2,6.096,6.091,6.453,6.014,5880000.0,0.003292,0.240537,0.109457,19720000.0,1.008282e+06,...,6.152852,2.711648,0.776401,0.072074,4,8,1,6.073,-0.003773,0.0
3,6.073,6.101,6.246,5.969,2640000.0,-0.003773,0.026712,0.016243,16250000.0,2.576438e+06,...,6.387083,2.703717,0.810350,0.045402,5,8,1,6.229,0.025687,1.0
4,6.229,6.077,6.237,5.944,3150000.0,0.025687,0.025181,0.015381,11670000.0,1.742154e+06,...,6.606805,2.747995,0.824990,0.048215,6,8,1,5.909,-0.051373,0.0


In [3]:
keep_as_is = ["last_return", "return_3d", "returnvola_3d", "return_7d", "returnvola_7d", "return_14d", "returnvola_14d",
              "return_21d", "returnvola_21d", "return_30d", "returnvola_30d", "return_60d", "returnvola_60d",
              "return_90d", "returnvola_90d", "rsi_14", "bb_width_20", "high_low_range_pct"]
feat = df[keep_as_is].copy()
print(len(keep_as_is), "relative columns kept")

18 relative columns kept


Dropped for now (levels): last_price, open, high, low, volume and its rolling sums, all SMAs/EMAs, MACD, Bollinger bands. We will re-create them as RATIOS

In [4]:
for w in [7, 14, 30, 50, 90, 200]:
    feat[f"price_to_sma_{w}"] = df["last_price"] / df[f"sma_{w}"] - 1
feat["ema_gap"] = df["ema_12"] / df["ema_26"] - 1
feat["macd_norm"] = df["macd"] / df["last_price"]
feat["macd_hist_norm"] = (df["macd"] - df["macd_signal"]) / df["last_price"]

In [5]:
feat["bb_position"] = (df["last_price"] - df["bb_lower_20"]) / (df["bb_upper_20"] - df["bb_lower_20"])
feat["body_pct"] = (df["last_price"] - df["open"]) / df["open"]
top = df[["open", "last_price"]].max(axis=1); bottom = df[["open", "last_price"]].min(axis=1)
feat["upper_wick"] = (df["high"] - top) / df["last_price"]
feat["lower_wick"] = (bottom - df["low"]) / df["last_price"]

In [6]:
feat["vol_ratio_7"] = df["volume"] / (df["volume_7d"] / 7)
feat["vol_ratio_30"] = df["volume"] / (df["volume_30d"] / 30)
feat["log_vol_change"] = np.log(df["volume"] / df["volume"].shift(1))

In [9]:
feat.head()

,last_return,return_3d,returnvola_3d,return_7d,returnvola_7d,return_14d,returnvola_14d,return_21d,returnvola_21d,return_30d,returnvola_30d,return_60d,returnvola_60d,return_90d,returnvola_90d,rsi_14,bb_width_20,high_low_range_pct,price_to_sma_7,price_to_sma_14,price_to_sma_30,price_to_sma_50,price_to_sma_90,price_to_sma_200,ema_gap,macd_norm,macd_hist_norm,bb_position,body_pct,upper_wick,lower_wick,vol_ratio_7,vol_ratio_30,log_vol_change
0,0.203704,0.121114,0.152823,0.427365,0.114407,0.628579,0.084346,0.474327,0.075195,0.528424,0.066129,1.060258,0.054241,1.339794,0.049782,77.367538,0.626104,0.218197,0.210920,0.342335,0.421249,0.588516,0.797775,0.852380,0.099984,0.072423,0.028004,1.146679,0.203949,0.011834,0.000000,1.311963,1.854137,NaN
1,0.027219,0.110989,0.153215,0.456376,0.113150,0.640832,0.084238,0.534731,0.074786,0.534731,0.066148,1.207047,0.053790,1.462910,0.049673,77.845576,0.714291,0.136956,0.178270,0.327797,0.435592,0.602507,0.824472,0.898056,0.114414,0.083178,0.031949,1.067612,0.028610,0.037525,0.067808,1.419465,2.248182,0.235182
2,0.003292,0.240537,0.109457,0.518306,0.108190,0.607171,0.084708,0.596647,0.073733,0.505185,0.066178,1.253604,0.053647,1.385910,0.049611,77.475543,0.776401,0.072074,0.117712,0.285940,0.417477,0.578866,0.809105,0.899613,0.123649,0.092119,0.032847,0.983479,0.000821,0.058563,0.012631,0.997818,1.656805,-0.273552
3,-0.003773,0.026712,0.016243,0.370262,0.109590,0.487023,0.084580,0.551213,0.073999,0.488845,0.066257,1.225357,0.053697,1.387186,0.049609,75.350497,0.810350,0.045402,0.067606,0.243812,0.390636,0.545285,0.781548,0.888115,0.128321,0.098359,0.031089,0.914729,-0.004589,0.023876,0.017125,0.441577,0.759275,-0.800778
4,0.025687,0.025181,0.015381,0.180629,0.090677,0.591873,0.081936,0.634908,0.073093,0.503137,0.066284,1.303624,0.053622,1.395769,0.049620,79.631525,0.824990,0.048215,0.069435,0.233954,0.404014,0.554910,0.805949,0.931129,0.132240,0.101300,0.028572,0.902093,0.025012,0.001284,0.021352,0.599185,0.907519,0.176624


In [10]:
for k in [1, 2, 3]:
    feat[f"ret_lag{k}"] = df["last_return"].shift(k)

In [11]:
feat["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7);  feat["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
feat["month_sin"] = np.sin(2 * np.pi * df["month"] / 12);     feat["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
feat["is_weekend"] = (df["day_of_week"] >= 5).astype(int)